# CAREBench — Notebook Launcher

Runs CAREBench from Google Colab, JupyterLab, or an HPC cluster without any local setup.

**Steps:**
1. Detect the environment and load an optional Hugging Face token.
2. Set the run options (the only cell you need to edit).
3. Clone the repository and install dependencies.
4. Run the selected track through `main.py`.
5. Optionally copy the run outputs to Google Drive (Colab only).

**Tracks:**
- `"reasoning"` — Track 3, the LLM context-shift assay. With `DRY_RUN = True` it uses deterministic mock responses and needs no GPU or token.
- `"representation"` — Track 1: embeddings, ensemble training with leave-one-patient-out CV, and statistical analysis.

By default both tracks run on the bundled **synthetic** example dataset. To use your own data, point `CONFIG_PATH` at a config whose paths reference it (see `docs/data_format.md`).

**Hugging Face token:** only needed for gated models when `DRY_RUN = False`. In Colab, add `HF_TOKEN` to the **Secrets** panel (key icon) and enable notebook access. Elsewhere, set the `HF_TOKEN` environment variable.

# 1. Environment

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import userdata
    IN_COLAB = True
    print("Environment: Google Colab")
except ImportError:
    userdata = None
    IN_COLAB = False
    print("Environment: Jupyter (non-Colab)")


def _get_secret(colab_key, env_keys):
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(colab_key)
            if val:
                return val
        except Exception:
            pass
    for k in env_keys:
        val = os.environ.get(k)
        if val:
            return val
    return None


HF_TOKEN = _get_secret("HF_TOKEN", ["HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"])
print("Hugging Face token:", "LOADED" if HF_TOKEN else "not set (fine for dry runs)")

# 2. Configuration

Edit the values below, then run the remaining cells.

In [ ]:
REPO_URL = "https://github.com/soroushdty/CAREBench.git"
BRANCH = "main"

# "reasoning" (Track 3, LLM assay) or "representation" (Track 1, training pipeline)
TRACK = "reasoning"

# Config file, relative to the repository root. None uses the track default:
#   reasoning      -> configs/assay_config.yaml
#   representation -> configs/main_config.yaml
CONFIG_PATH = None

# Track 3 options
DRY_RUN = True    # True: mock LLM responses, no GPU or token needed
MODEL_ID = None   # e.g. "Qwen/Qwen2.5-14B-Instruct"; None runs the config's model_ids

# Where to clone the repository. Override if your HPC home layout differs.
CLONE_ROOT = Path("/content") if IN_COLAB else Path.home() / "repos"

# Google Drive (Colab only)
SAVE_TO_DRIVE = False
DRIVE_FOLDER = "CAREBench"

# 3. Clone and install

In [ ]:
clone_dir = CLONE_ROOT / "CAREBench"

if clone_dir.exists():
    print(f"Removing existing checkout at {clone_dir} ...")
    shutil.rmtree(clone_dir)

print(f"Cloning {REPO_URL} (branch {BRANCH}) into {clone_dir} ...")
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(clone_dir)],
    check=True,
)

print("Installing dependencies (this can take a few minutes) ...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(clone_dir / "requirements.txt")],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(clone_dir)],
    check=True,
)
print(f"Ready: {clone_dir}")

# 4. Run

In [ ]:
if TRACK not in {"reasoning", "representation"}:
    raise ValueError(f"TRACK must be 'reasoning' or 'representation', got {TRACK!r}")

default_configs = {
    "reasoning": "configs/assay_config.yaml",
    "representation": "configs/main_config.yaml",
}
config_path = CONFIG_PATH or default_configs[TRACK]

cmd = [sys.executable, "main.py", "--track", TRACK, "--", "--config", config_path]
if TRACK == "reasoning":
    if DRY_RUN:
        cmd.append("--dry_run")
    if MODEL_ID:
        cmd += ["--model", MODEL_ID]
    elif DRY_RUN:
        cmd += ["--model", "dry_run_example"]

env = os.environ.copy()
env["PYTHONHASHSEED"] = "42"  # must be set before the Python process starts
if HF_TOKEN:
    env["HF_TOKEN"] = env["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=str(clone_dir), env=env)
if result.returncode != 0:
    raise RuntimeError(f"main.py exited with code {result.returncode}")
print("Run complete.")

# 5. Save outputs to Google Drive (Colab only)

Copies the most recent run folder (`output/run_*` for Track 1, `output/assay/run_*` for Track 3) to `MyDrive/<DRIVE_FOLDER>/`.

In [ ]:
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    output_root = clone_dir / ("output/assay" if TRACK == "reasoning" else "output")
    run_folders = sorted(p for p in output_root.glob("run_*") if p.is_dir())
    if run_folders:
        src = run_folders[-1]
        dst = Path("/content/drive/MyDrive") / DRIVE_FOLDER / src.name
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"Copied {src} -> {dst}")
    else:
        print(f"No run folders found under {output_root}.")
else:
    print("Skipping Google Drive copy (not in Colab, or SAVE_TO_DRIVE is False).")